In [1]:
import pandas as pd
df = pd.read_csv('../data/raw/vedem/V-DEM-CY-Core-v16.csv')

In [2]:
country = ['CHN', 'TWN', 'JPN', 'KOR', 'PRK', 'PHL', 'VNM']
df_clean = df.loc[
    (df['country_text_id'].isin(country)) & (df['year'] >= 2015) & (df['year'] <= 2025),
    ['country_text_id', 'v2x_libdem', 'year']
]
print(df_clean['country_text_id'].unique())
print(df_clean.groupby('country_text_id')['year'].count())

<ArrowStringArray>
['JPN', 'VNM', 'PRK', 'KOR', 'PHL', 'TWN', 'CHN']
Length: 7, dtype: str
country_text_id
CHN    11
JPN    11
KOR    11
PHL    11
PRK    11
TWN    11
VNM    11
Name: year, dtype: int64


In [3]:
dyad_list = [
    ('CHN', 'TWN'), ('CHN', 'JPN'), ('CHN', 'KOR'), ('CHN', 'PRK'),
    ('JPN', 'KOR'), ('JPN', 'PRK'), ('JPN', 'TWN'),
    ('KOR', 'PRK'), ('KOR', 'TWN'),
    ('PRK', 'TWN'),
    ('CHN', 'PHL'), ('CHN', 'VNM'),
]

years = range(2015, 2026)

vdem_dyad_list = []
for actor1, actor2 in dyad_list:
    for year in years:
        vdem_dyad_list.append({'dyad': f'{actor1}-{actor2}', 'year': year, 'country1': actor1, 'country2': actor2})

vdem_dyad = pd.DataFrame(vdem_dyad_list)

In [7]:
vdem_dyad = pd.merge(
    vdem_dyad,
    df_clean,
    left_on=['country1', 'year'],
    right_on=['country_text_id', 'year'],
    how='left'
)
vdem_dyad = vdem_dyad.rename(columns={'v2x_libdem': 'libdem_1'})
vdem_dyad = vdem_dyad.drop(columns=['country_text_id'])

In [10]:
vdem_dyad = pd.merge(
    vdem_dyad,
    df_clean,
    left_on=['country2', 'year'],
    right_on=['country_text_id', 'year'],
    how='left'
)
vdem_dyad = vdem_dyad.rename(columns={'v2x_libdem': 'libdem_2'})
vdem_dyad = vdem_dyad.drop(columns=['country_text_id'])

In [21]:
vdem_dyad.info()

<class 'pandas.DataFrame'>
RangeIndex: 132 entries, 0 to 131
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   dyad         132 non-null    str    
 1   year         132 non-null    int64  
 2   country1     132 non-null    str    
 3   country2     132 non-null    str    
 4   libdem_1     132 non-null    float64
 5   libdem_2     132 non-null    float64
 6   regime_diff  132 non-null    float64
dtypes: float64(3), int64(1), str(3)
memory usage: 9.0 KB


In [17]:
vdem_dyad['regime_diff'] = abs(vdem_dyad['libdem_1'] - vdem_dyad['libdem_2'])
vdem_dyad.to_csv('../data/processed/vdem_dyad.csv')

In [24]:
df_gdelt = pd.read_csv('../data/processed/modelA_dataset.csv')
df_gdelt['year'] = df_gdelt['MonthYear'] // 100
df_gdelt.info()

<class 'pandas.DataFrame'>
RangeIndex: 1440 entries, 0 to 1439
Data columns (total 28 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   dyad                      1440 non-null   str    
 1   MonthYear                 1440 non-null   int64  
 2   event_count               1440 non-null   float64
 3   goldstein_std             1440 non-null   float64
 4   goldstein_min             1440 non-null   float64
 5   num_mentions_sum          1440 non-null   float64
 6   num_articles_sum          1440 non-null   float64
 7   num_sources_sum           1440 non-null   float64
 8   high_conflict_count       1440 non-null   float64
 9   low_conflict_count        1440 non-null   float64
 10  quad4_count               1440 non-null   float64
 11  high_conflict_pct         1440 non-null   float64
 12  low_conflict_pct          1440 non-null   float64
 13  quad4_pct                 1440 non-null   float64
 14  event_count_lag1   

In [ ]:
final_dataset = pd.merge(
    vdem_dyad[['dyad', 'year', 'regime_diff']],
    df_gdelt,
    on = ['dyad', 'year'],
    how = 'right'
)
final_dataset.head()

,dyad,year,regime_diff,MonthYear,event_count,goldstein_std,goldstein_min,num_mentions_sum,num_articles_sum,num_sources_sum,...,num_mentions_sum_lag1,num_articles_sum_lag1,num_sources_sum_lag1,high_conflict_count_lag1,low_conflict_count_lag1,quad4_count_lag1,high_conflict_pct_lag1,low_conflict_pct_lag1,quad4_pct_lag1,monthly_label
0,CHN-JPN,2015,0.688,201502,1342.0,4.358883,-10.0,6360.0,6303.0,1405.0,...,543.0,529.0,115.0,2.0,17.0,3.0,0.017391,0.147826,0.026087,Cooperation
1,CHN-JPN,2015,0.688,201503,7929.0,3.837797,-10.0,35727.0,35260.0,8537.0,...,6360.0,6303.0,1405.0,99.0,330.0,142.0,0.073770,0.245902,0.105812,Cooperation
2,CHN-JPN,2015,0.688,201504,6904.0,4.077461,-10.0,31897.0,31518.0,7633.0,...,35727.0,35260.0,8537.0,271.0,2030.0,407.0,0.034178,0.256022,0.051331,Cooperation
3,CHN-JPN,2015,0.688,201505,5125.0,4.444714,-10.0,22334.0,22019.0,5374.0,...,31897.0,31518.0,7633.0,320.0,1783.0,501.0,0.046350,0.258256,0.072567,Cooperation
4,CHN-JPN,2015,0.688,201506,5552.0,4.225468,-10.0,24224.0,23946.0,6041.0,...,22334.0,22019.0,5374.0,335.0,1416.0,482.0,0.065366,0.276293,0.094049,Cooperation


In [30]:
print(final_dataset['regime_diff'].isna().sum())
print(final_dataset.shape)
final_dataset.to_csv('../data/processed/modelB_dataset.csv', index = False)

0
(1440, 29)
